In [1]:
import os

In [2]:
bucket_name = "pair-email-classification"
da_with_invoice_object_base = "data/aftercourt/drittauskunf_with_invoice/"
da_without_invoice_object_base = "data/aftercourt/drittauskunft_without_invoice/"
da_with_invoice_local_path = "/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/assets/pdfs/drittauskunft/with_invoice"
da_without_invoice_local_path = "/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/assets/pdfs/drittauskunft/without_invoice"

In [3]:
da_invoice_local_paths = [os.path.join(da_with_invoice_local_path, filename) for filename in os.listdir(da_with_invoice_local_path) if filename.endswith(".pdf")]
da_no_invoice_local_paths = [os.path.join(da_without_invoice_local_path, filename) for filename in os.listdir(da_without_invoice_local_path) if filename.endswith(".pdf")]

In [4]:
print(f"Found {len(da_invoice_local_paths)} PDFs with invoice and {len(da_no_invoice_local_paths)} PDFs without invoice.") 

Found 26 PDFs with invoice and 27 PDFs without invoice.


In [5]:
print(da_invoice_local_paths[0], "\n", da_no_invoice_local_paths[0])

/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/assets/pdfs/drittauskunft/with_invoice/0_sammel120251218-309-jspu2c.pdf 
 /Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/assets/pdfs/drittauskunft/without_invoice/Dr_ii_635_25_an_gl_mitt_ergebnis_drittauskunft_ohne_hb20251219-311-1savlpi.pdf


In [6]:
import sys
sys.path.append("/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation")
from utils.use_textract_utils import parse_local_pdfs_with_textract

In [7]:
da_invoice_texts = parse_local_pdfs_with_textract(
    local_pdf_paths=da_invoice_local_paths,
    s3_object_key_base=da_with_invoice_object_base,
    s3_bucket_name=bucket_name,
    use_page_markers=True,
)

In [8]:
da_no_invoice_texts = parse_local_pdfs_with_textract(
    local_pdf_paths=da_no_invoice_local_paths,
    s3_object_key_base=da_without_invoice_object_base,
    s3_bucket_name=bucket_name,
    use_page_markers=True,
)

In [9]:
import pandas as pd

output_dir = "/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/notebooks/classification/drittauskunft/dataset"

rows = [
    {"s3_key": k, "text": v, "label": "with_invoice"} for k, v in da_invoice_texts.items()
] + [
    {"s3_key": k, "text": v, "label": "without_invoice"} for k, v in da_no_invoice_texts.items()
]

df = pd.DataFrame(rows)
df.to_parquet(os.path.join(output_dir, "drittauskunft_objectkey_to_texts.parquet"), index=False)
print(df["label"].value_counts())

label
without_invoice    27
with_invoice       26
Name: count, dtype: int64


In [10]:
df

,s3_key,text,label
0,data/aftercourt/drittauskunf_with_invoice/0_sa...,<page_1>\nObergerichtsvollzieher Frank Schäfer...,with_invoice
1,data/aftercourt/drittauskunf_with_invoice/comb...,<page_1>\nAnlage zu GZ: St II 4 S 0229a KEVIZZ...,with_invoice
2,data/aftercourt/drittauskunf_with_invoice/Doku...,<page_1>\nThomas Vetter\nJohann-Ruland-Weg 9\n...,with_invoice
3,data/aftercourt/drittauskunf_with_invoice/comb...,<page_1>\nBundeszentralamt\nfür Steuern\nPOSTA...,with_invoice
4,data/aftercourt/drittauskunf_with_invoice/63_d...,<page_1>\nChristian Linner\nSchragenhofstraße ...,with_invoice
5,data/aftercourt/drittauskunf_with_invoice/1_dr...,<page_1>\nGerichtsvollzieherin\nWaldhofer Stra...,with_invoice
6,data/aftercourt/drittauskunf_with_invoice/1_dr...,<page_1>\nGerichtsvollzieherin\nWaldhofer Stra...,with_invoice
7,data/aftercourt/drittauskunf_with_invoice/Doku...,<page_1>\nDaniel Arlt\nUlrich-Kaegerl-Straße 5...,with_invoice
8,data/aftercourt/drittauskunf_with_invoice/comb...,<page_1>\nObergerichtsvollzieherin\nAmtsgerich...,with_invoice
9,data/aftercourt/drittauskunf_with_invoice/B_DR...,<page_1>\nObergerichtsvollzieherin Evelyn Häp\...,with_invoice


In [11]:
print(df.iloc[0]["text"])

<page_1>
Obergerichtsvollzieher Frank Schäfer
bei dem Amtsgericht Darmstadt
Mein Zeichen:
DR II 1870/25
OGV Frank Schäfer, Mathildenplatz 12, 64283 Darmstadt
Bitte immer angeben!
Firma
Ihr Zeichen: 177398801833
PAIR Finance GmbH
Knesebeckstraße 62-63
Telefon 06150/55301,Mobil:017692264261
10719 Berlin
Telefax 06150/591668
Erzhausen, 18.12.2025
Zwangsvollstreckungssache
Firma Liquandum Capital II GmbH, Knesebeckstraße 62-63, 10719 Berlin
vertr. d.
Firma PAIR Finance GmbH, Knesebeckstraße 62-63, 10719 Berlin
gegen
Herrn Labid Ahmad, Falkenweg 7, 64331 Weiterstadt
Sehr geehrte Damen und Herren,
in oben genannter Sache teile ich Ihnen mit, dass im Termin zur Abgabe der Vermögensauskunftder Schuldner trotz
ordnungsgemäßer Ladung nicht erschienen ist.
Erlass eines Haftbefehls wurde nicht beantragt
Nach Zweckerreichung sind erlangte Schuldnerdaten zu löschen.
Die entstandenen Kosten bitte ich auf mein Dienstkonto zu überweisen. GiroCode 75,11€.
Kostenrechnung GvKostG (KV=Kostenverzeichnis) in

# add new data to final_raw_dataset.csv

In [13]:
raw_data = pd.read_csv('/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/data/raw/final_raw_data.csv')
raw_data.columns

Index(['ticket_uuid', 'attachment_id', 'text', 'object_key', 'document_type',
       'cleaned_text', 'data', 'is_pfub', 'is_ladung', 's3_link',
       'textract_job_id', 'textract_s3_link', 'is_ve_with_invoice'],
      dtype='object')

In [21]:
raw_data['is_ladung'].dtype

dtype('bool')

In [22]:
raw_data['is_pfub'].dtype

dtype('bool')

alter df (new data) to match the fields.

In [ ]:
import re
df.rename(columns={"s3_key": "object_key"}, inplace=True)
df.rename(columns={"text": "text_w_pages"}, inplace=True)
# remove page markers
page_marker_pattern = r"<page_\d+>\n"
df['text'] = df['text_w_pages'].apply(lambda x: re.sub(page_marker_pattern, " ", x))

In [16]:
df.rename(columns={"label": "is_da_with_invoice"}, inplace=True)

In [18]:
df['is_da_with_invoice'] = df['is_da_with_invoice'].apply(lambda x: True if x == "with_invoice" else False)

In [19]:
df

,object_key,text_w_pages,is_da_with_invoice,text
0,data/aftercourt/drittauskunf_with_invoice/0_sa...,<page_1>\nObergerichtsvollzieher Frank Schäfer...,True,Obergerichtsvollzieher Frank Schäfer\nbei dem...
1,data/aftercourt/drittauskunf_with_invoice/comb...,<page_1>\nAnlage zu GZ: St II 4 S 0229a KEVIZZ...,True,Anlage zu GZ: St II 4 S 0229a KEVIZZ 2948091/...
2,data/aftercourt/drittauskunf_with_invoice/Doku...,<page_1>\nThomas Vetter\nJohann-Ruland-Weg 9\n...,True,Thomas Vetter\nJohann-Ruland-Weg 9\nGerichtsv...
3,data/aftercourt/drittauskunf_with_invoice/comb...,<page_1>\nBundeszentralamt\nfür Steuern\nPOSTA...,True,Bundeszentralamt\nfür Steuern\nPOSTANSCHRIFT\...
4,data/aftercourt/drittauskunf_with_invoice/63_d...,<page_1>\nChristian Linner\nSchragenhofstraße ...,True,Christian Linner\nSchragenhofstraße 27\nGeric...
5,data/aftercourt/drittauskunf_with_invoice/1_dr...,<page_1>\nGerichtsvollzieherin\nWaldhofer Stra...,True,Gerichtsvollzieherin\nWaldhofer Straße 17\nKa...
6,data/aftercourt/drittauskunf_with_invoice/1_dr...,<page_1>\nGerichtsvollzieherin\nWaldhofer Stra...,True,Gerichtsvollzieherin\nWaldhofer Straße 17\nKa...
7,data/aftercourt/drittauskunf_with_invoice/Doku...,<page_1>\nDaniel Arlt\nUlrich-Kaegerl-Straße 5...,True,Daniel Arlt\nUlrich-Kaegerl-Straße 5\nOberger...
8,data/aftercourt/drittauskunf_with_invoice/comb...,<page_1>\nObergerichtsvollzieherin\nAmtsgerich...,True,Obergerichtsvollzieherin\nAmtsgericht Wolfsbu...
9,data/aftercourt/drittauskunf_with_invoice/B_DR...,<page_1>\nObergerichtsvollzieherin Evelyn Häp\...,True,Obergerichtsvollzieherin Evelyn Häp\nbeim Amt...


In [23]:
df['document_type'] = 'drittauskunft'
df['is_ladung'] = False
df['is_pfub'] = False

save the data

In [27]:
df.to_csv(os.path.join(output_dir, "add_data_drittauskunft.csv"), index=False)

add new data using update raw data script

In [28]:
from src.data_handling.update_raw_data import update_raw_data

updated_raw_data = update_raw_data(raw_data, df)
print(updated_raw_data.shape)
updated_raw_data.tail()

Starting raw data update. Raw data: 5532 rows, New data: 53 rows.
Validating required columns: ['text', 'document_type', 'is_pfub', 'is_ladung']
All required columns validated successfully.
Filling generated columns for 53 rows.
Validating dtypes between raw data and new data.
No duplicates found.
Raw data update complete. Total rows: 5585 (added 53).
(5585, 15)


,ticket_uuid,attachment_id,text,object_key,document_type,cleaned_text,data,is_pfub,is_ladung,s3_link,textract_job_id,textract_s3_link,is_ve_with_invoice,text_w_pages,is_da_with_invoice
5580,4f6c9081-0928-527c-9356-943ba2237293,e3a0150a-cdb8-5a9c-be4a-0f5603361356,R. Müller\nObergerichtsvollzieher\nLerchenweg...,data/aftercourt/drittauskunft_without_invoice/...,drittauskunft,NaN,NaN,False,False,NaN,fe02c9970e549b20806d238d71c0adfe8a6f3dccdf820a...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,<page_1>\nR. Müller\nObergerichtsvollzieher\nL...,False
5581,50143302-3ac3-587b-ae25-047d34d94750,99d61bd2-665c-59b1-a24d-264f41c2f474,Bundeszentralamt\nfür Steuern\nPOSTANSCHRIFT\...,data/aftercourt/drittauskunft_without_invoice/...,drittauskunft,NaN,NaN,False,False,NaN,93e56ec1cfb7719bf463bf44585e833cdd56fb9689d4f3...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,<page_1>\nBundeszentralamt\nfür Steuern\nPOSTA...,False
5582,de97e37c-ebf7-5c0e-92be-209db058d9ae,60913326-3481-573b-a96e-72bb2ca9a9ef,Anne-Katrin Götz\nAmtsgericht Dortmund\nOberg...,data/aftercourt/drittauskunft_without_invoice/...,drittauskunft,NaN,NaN,False,False,NaN,870b8fd6e0da13bafd14ed6e5cec7e79af8c7ae43cae66...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,<page_1>\nAnne-Katrin Götz\nAmtsgericht Dortmu...,False
5583,c72625b3-d66a-5df5-a933-11f57215838a,2a6e990a-3458-5efe-a8d9-f8b988af1740,Obergerichtsvollzieherin G. Samuels\nHESSEN\n...,data/aftercourt/drittauskunft_without_invoice/...,drittauskunft,NaN,NaN,False,False,NaN,6eb9335329883f671aa167608aa4108d45a5e0fcbbbb86...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,<page_1>\nObergerichtsvollzieherin G. Samuels\...,False
5584,9d233a96-2fd2-5f56-ab60-58043398223d,f8f00142-5ef8-5940-8ec0-f211617274d8,Gerichtsvollzieherin\nKreuzgasse 17\nL. Lutz\...,data/aftercourt/drittauskunft_without_invoice/...,drittauskunft,NaN,NaN,False,False,NaN,0d1359e435b18356222e26c4fc1eeb79c53373771c782b...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,<page_1>\nGerichtsvollzieherin\nKreuzgasse 17\...,False


In [26]:
updated_raw_data.document_type.value_counts()

document_type
ladung_va                                 2030
mail_attachments                           984
monierung_mb                               545
attachment_and_transfer_order              535
approved_seizure                           523
court_inbox                                346
fp_protocol                                220
fp_invoice                                 128
vermögensverzeichnis                        89
approved_attachment_and_transfer_order      64
drittauskunft                               53
enforcement_order                           23
tbd                                         22
bailiff_ip                                  20
contradiction                                3
Name: count, dtype: int64